In [ ]:
# === Setup ===
# Runtime: <1m with OAI_FAST_MODE=1
# Hardware: CPU smoke
# Network: none
# Competition-safe: Yes for the declared profile
import os, random, math, re, json, csv, time
from pathlib import Path
import numpy as np
FAST_MODE = os.getenv("OAI_FAST_MODE", "0") == "1"
RUNTIME_PROFILE = os.getenv("OAI_RUNTIME_PROFILE", "cpu")
random.seed(42)
np.random.seed(42)
print(f"Runtime profile: {'fast' if FAST_MODE else 'full'}")

# Reverse-mode autograd từ số 0

Graph lưu phép toán; backward duyệt topo ngược và cộng gradient khi một node có nhiều nhánh.

In [ ]:
class Value:
    """Scalar reverse-mode autodiff node."""
    def __init__(self, data, children=(), op=""):
        self.data = float(data); self.grad = 0.0
        self._prev = set(children); self._op = op; self._backward = lambda: None
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def backward(): self.grad += out.grad; other.grad += out.grad
        out._backward = backward; return out
    __radd__ = __add__
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def backward(): self.grad += other.data*out.grad; other.grad += self.data*out.grad
        out._backward = backward; return out
    __rmul__ = __mul__
    def __pow__(self, power):
        out = Value(self.data**power, (self,), f"**{power}")
        def backward(): self.grad += power*self.data**(power-1)*out.grad
        out._backward = backward; return out
    def tanh(self):
        t = math.tanh(self.data); out = Value(t, (self,), "tanh")
        def backward(): self.grad += (1-t*t)*out.grad
        out._backward = backward; return out
    def backward(self):
        topo=[]; seen=set()
        def visit(node):
            if node not in seen:
                seen.add(node)
                for parent in node._prev: visit(parent)
                topo.append(node)
        visit(self); self.grad=1.0
        for node in reversed(topo): node._backward()

x, w, b = Value(2.0), Value(-3.0), Value(1.0)
y = (x*w + b).tanh(); y.backward()
print("y, dx, dw, db:", y.data, x.grad, w.grad, b.grad)

In [ ]:
def f(x): return math.tanh(2*x*x - 3*x + 1)
x0=0.7; eps=1e-6
numeric=(f(x0+eps)-f(x0-eps))/(2*eps)
x=Value(x0); out=(2*x*x + (-3)*x + 1).tanh(); out.backward()
assert abs(x.grad-numeric) < 1e-5
print("gradient check:", x.grad, numeric)